# آموزش موازی PPO برای هاور — ۴۰ میلیون گام محیطی

این نوت‌بوک **یک مدل مشترک PPO** را با تجربه‌های حاصل از **۴۰ محیط شبیه‌سازی موازی** آموزش می‌دهد؛ ۴۰ مدل جدا ساخته نمی‌شود. هدف، مجموع ۴۰٬۰۰۰٬۰۰۰ انتقال محیطی است. در هر به‌روزرسانی، ۴۰ محیط × ۱٬۰۰۰ گام = ۴۰٬۰۰۰ نمونه جمع می‌کنند و همان یک مدل به‌روزرسانی می‌شود.

اکشن‌ها در بازهٔ متقارن ‎[-1, 1]‎ هستند و گاز به بازهٔ عملیاتی شبیه‌ساز تبدیل می‌شود. **محدودیت:** این فقط آموزش شبیه‌سازی است؛ مدل فیزیکی واقعی و هم‌ارزی عددی کامل هنوز تأیید نشده‌اند و مدل برای پرواز واقعی آماده یا تأییدشده نیست. این تنظیم تعداد محیط‌ها مطابق تنظیم اصلی پروژه روی ۴۰ حفظ شده است. شبکهٔ PPO روی CUDA اجرا می‌شود، اما هستهٔ فیزیک C++ فعلی هنوز CPUمحور است؛ انتقال فیزیک به GPU نیازمند پورت و آزمون مستقل است.

In [ ]:
!git clone -b feat/cuda-training-diagnostics https://github.com/samanramezani1377-hub/3d-fly-test.git
%cd /content/3d-fly-test
!python -m pip install -r training/requirements.txt


## ساخت هستهٔ فیزیک C++

اگر این مرحله خطا داد، آموزش را اجرا نکن و متن خطا را بفرست.

In [ ]:
!python training/build_native_core.py


## اجرای آموزش اصلی

تنظیم فعلی یک **سیاست PPO مشترک** و ۴۰ محیط موازی دارد. تعداد ۴۰ میلیون، مجموع انتقال‌های همهٔ محیط‌هاست؛ نه ۴۰ میلیون برای هر محیط. checkpoint و ارزیابی هر ۴۰۰٬۰۰۰ انتقال کل انجام می‌شوند. اجرای کامل ممکن است در یک نشست Colab تمام نشود؛ ZIP خروجی را پیش از قطع نشست دانلود کن. اگر منابع کافی نبود، `N_ENVS` را کم کن.

In [ ]:
TIMESTEPS = 40_000_000
N_ENVS = 40
N_STEPS = 1_000
BATCH_SIZE = 4_000
RUN_NAME = 'ppo-hover-cuda-40m'
!python training/train_ppo.py --timesteps {TIMESTEPS} --n-envs {N_ENVS} --n-steps {N_STEPS} --batch-size {BATCH_SIZE} --device cuda --seed 42 --run-name {RUN_NAME} --save-freq 400000 --eval-freq 400000 --eval-episodes 10


## ارزیابی مدل نهایی

این ارزیابی بعد از پایان آموزش اجرا می‌شود و نتیجه را داخل پوشهٔ اجرا ذخیره می‌کند.

In [ ]:
!python training/evaluate.py --model training/runs/{RUN_NAME}/final_model.zip --episodes 10 --output training/runs/{RUN_NAME}/final-evaluation.json
!if [ -f training/runs/{RUN_NAME}/best/best_model.zip ]; then python training/evaluate.py --model training/runs/{RUN_NAME}/best/best_model.zip --episodes 10 --output training/runs/{RUN_NAME}/best-evaluation.json; fi


## دانلود همهٔ خروجی‌ها

ZIP شامل مدل نهایی، بهترین مدل ارزیابی‌شده، checkpointها، لاگ‌ها و گزارش‌های ارزیابی است. آن را قبل از بستن Colab دانلود کن.

In [ ]:
!cd training/runs && zip -qr /content/{RUN_NAME}-results.zip {RUN_NAME}
from google.colab import files
files.download(f'/content/{RUN_NAME}-results.zip')


## نکتهٔ مهم برای ادامهٔ کار

۴۰ میلیون گام آموزش، موفقیت را تضمین نمی‌کند. معیار تصمیم، گزارش ارزیابی، نرخ برخورد، موفقیت هاور، سهم زمان پایدار، خطای ارتفاع و سرعت عمودی است. اگر نتیجه خوب نبود، ZIP را در همین گفتگو بفرست تا مرحلهٔ بعد را بر اساس گزارش‌ها اصلاح کنیم. این مدل را روی کوادکوپتر واقعی اجرا نکن.